Zero-shot using deepseek

In [ ]:
from groq import Groq
import time
import json
import pandas as pd
import re

# Initialize Groq client
groq_client = Groq(api_key="")

# Load JSON file
with open('/content/test_collection_IPC_201.json', 'r') as f:
    data = json.load(f)

# Process all documents
case_data = []
for filename, case_details in data.items():
    case_data.append({
        "doc_id": filename,
        "fact": case_details.get("fact", "")
    })

test_df = pd.DataFrame(case_data)

def model_groq(model_name, num_tokens, sp, up):
    while True:
        try:
            completion = groq_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": sp}, {"role": "user", "content": up}],
                temperature=0,
                max_tokens=num_tokens,
                top_p=0.0,
                stream=False,
                stop=None,
            )
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 2 minutes...")
            time.sleep(120)
    return completion.choices[0].message.content

model_name = "deepseek-r1-distill-llama-70b"
num_tokens = 7998

prompt = """Analyze the given legal case fact pattern and identify ALL applicable IPC sections from:
"Indian Penal Code 498A": " Whoever, being the husband or the relative of the husband of a woman, subjects such woman to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine.",
"Indian Penal Code 506": " Whoever commits the offence of criminal intimidation shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both",
"Indian Penal Code 147": " Whoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.",
"Indian Penal Code 201": " Whoever, knowing or having reason to believe that an offence has been committed, causes any evidence of the commission of that offence to disappear, with the intention of screening the offender from legal punishment, or with that intention gives any information respecting the offence which he knows or believes to be false;",
"Indian Penal Code 302": " Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 376": " Whoever, except in the cases provided for in sub-section (2), commits rape, shall be punished with rigorous imprisonment of either description for a term which shall not be less than ten years, but which may extend to imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 420": " Whoever cheats and thereby dishonestly induces the person deceived to deliver any property to any person, or to make, alter or destroy the whole or any part of a valuable security, or anything which is signed or sealed, and which is capable of being converted into a valuable security, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine."


Return ONLY the full statute names in list format. Example: ["Indian Penal Code 302", "Indian Penal Code 147"."Indian Penal Code 420","Indian Penal Code 498A","Indian Penal Code 506","Indian Penal Code 201","Indian Penal Code 376"]"""

def format_sections(response):
    # Extract full statute names using regex
    pattern = r'"Indian Penal Code (302|147|376|498A|506|201|420)"'
    matches = re.findall(pattern, response)

    # Format with full statute names
    formatted = [f"Indian Penal Code {section}"
                 for section in matches]

    # Remove duplicates while preserving order
    seen = set()
    return [x for x in formatted if not (x in seen or seen.add(x))]

def main():
    results = {}

    for i in range(len(test_df)):  # Process all documents
        doc_id = test_df.iloc[i]['doc_id']
        fact_text = test_df.iloc[i]['fact']

        response = model_groq(model_name, num_tokens,
                            prompt,
                            f"Case Facts: {fact_text}")

        sections = format_sections(response)
        results[doc_id] = sections

        print(f"Processed {doc_id}")
        print(f"Identified sections: {sections}\n")

    # Save results
    with open("ipc_sections_deepseek_zeroshot.json", "w") as f:
        json.dump(results, f, indent=4)
    print("Final results saved to ipc_sections_zeroshot.json")

if __name__ == "__main__":
    main()

Processed 2008.INSC.11.txt
Identified sections: ['Indian Penal Code 498A', 'Indian Penal Code 506', 'Indian Penal Code 201', 'Indian Penal Code 302']

Processed 2012.INSC.535.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 498A', 'Indian Penal Code 201', 'Indian Penal Code 420', 'Indian Penal Code 506']

Processed 2003.INSC.624.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201']

Processed 1997.INSC.784.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 506']

Processed 1996.INSC.1253.txt
Identified sections: ['Indian Penal Code 201']

Processed 1996.INSC.1218.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201', 'Indian Penal Code 506']

Processed 2009.INSC.355.txt
Identified sections: ['Indian Penal Code 201', 'Indian Penal Code 302']

Processed 2014.INSC.683.txt
Identified sections: ['Indian Penal Code 420']

Processed 2013.INSC.348.txt
Identified sections: ['Indian Penal Code 302']

Processed 200

KeyboardInterrupt: 

Few-shot using deepseek

In [ ]:
from groq import Groq
import time
import json
import pandas as pd
import re

# Initialize Groq client
groq_client = Groq(api_key="")

# Load JSON file
with open('/content/test_collection_IPC_201.json', 'r') as f:
    data = json.load(f)

# Process first 20 documents
case_data = []
for filename, case_details in list(data.items())[:50]:
    case_data.append({
        "doc_id": filename,
        "fact": case_details.get("fact", "")
    })

test_df = pd.DataFrame(case_data)

def model_groq(model_name, num_tokens, sp, up):
    while True:
        try:
            completion = groq_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": sp}, {"role": "user", "content": up}],
                temperature=0,
                max_tokens=num_tokens,
                top_p=0.0,
                stream=False,
                stop=None,
            )
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 2 minutes...")
            time.sleep(120)
    return completion.choices[0].message.content

model_name = "deepseek-r1-distill-llama-70b"
num_tokens = 6000

# Updated prompt with few-shot examples
prompt = """ Analyze the given legal case fact pattern and identify ALL applicable IPC sections from:
"Indian Penal Code 498A": " Whoever, being the husband or the relative of the husband of a woman, subjects such woman to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine.",
"Indian Penal Code 506": " Whoever commits the offence of criminal intimidation shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both",
"Indian Penal Code 147": " Whoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.",
"Indian Penal Code 201": " Whoever, knowing or having reason to believe that an offence has been committed, causes any evidence of the commission of that offence to disappear, with the intention of screening the offender from legal punishment, or with that intention gives any information respecting the offence which he knows or believes to be false;",
"Indian Penal Code 302": " Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 376": " Whoever, except in the cases provided for in sub-section (2), commits rape, shall be punished with rigorous imprisonment of either description for a term which shall not be less than ten years, but which may extend to imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 420": " Whoever cheats and thereby dishonestly induces the person deceived to deliver any property to any person, or to make, alter or destroy the whole or any part of a valuable security, or anything which is signed or sealed, and which is capable of being converted into a valuable security, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine."


Return ONLY the full statute names in list format. Example: ["Indian Penal Code 302", "Indian Penal Code 147"."Indian Penal Code 420","Indian Penal Code 498A","Indian Penal Code 506","Indian Penal Code 201","Indian Penal Code 376"]


Examples:

Example 1:
Case Facts: As there were allegations prima facie showing that the witnesses have been threatened, a ground for cancellation of bail did exist.
Applicable IPC Sections: ["Indian Penal Code 147"]

Example 2:
Case Facts: He went there and saw the two servants, throwing sticks on the fire and burring the dead body of Sudha Onkar Singh and Santosh were also present.No pyre was made and the dead body was burnt by sticks.
Applicable IPC Sections: ["Indian Penal Code 201"]

Example 3:
Case Facts: they armed with weapons, stopped the bus going from Dholpur to Khuthiyana Ghat, asked the passengers to get down attempted to drag out Ram Babu conductor of the bus and then appellant Rammo by firing two shots from his gun and others by their weapons injured and thereby killed Ram Babu.
Applicable IPC Sections: ["Indian Penal Code 302"]

Example 4:
Case Facts: committed sexual intercourse with her against her will and consent and thereafter, the rest of the accused had also committed rape on her.
Applicable IPC Sections: ["Indian Penal Code 376"]

Example 5:
Case Facts: he accused had entered into a criminal conspiracy, the ultimate object of which was to misappropriate dishonestly the charitable estate and converting the said estate to their own use.

Applicable IPC Sections: ["Indian Penal Code 420"]


Example 6:
Case Facts: Kalpana informed her parents that she has been harassed to get the balance amount by her husband and his relatives.

Applicable IPC Sections: ["Indian Penal Code 498A"]

Example 7:
Case Facts: the first respondent and his friend Dinesh Chaudhary assaulted the appellant and held a revolver against the chest of the appellant and threatened him.

Applicable IPC Sections: ["Indian Penal Code 506"]

Now analyze the following new case. Return ONLY the full statute names in list format."""

def format_sections(response):
    pattern = r'"Indian Penal Code (302|147|376|498A|506|201|420)"'
    matches = re.findall(pattern, response)
    formatted = [f"Indian Penal Code {section}" for section in matches]
    seen = set()
    return [x for x in formatted if not (x in seen or seen.add(x))]

def main():
    results = {}

    for i in range(min(50, len(test_df))):
        doc_id = test_df.iloc[i]['doc_id']
        fact_text = test_df.iloc[i]['fact']

        response = model_groq(model_name, num_tokens,
                            prompt,
                            f"Case Facts: {fact_text}")

        sections = format_sections(response)
        results[doc_id] = sections

        print(f"Processed {doc_id}")
        print(f"Identified sections: {sections}\n")

    # Save results
    with open("ipc_sections_fewshot.json", "w") as f:
        json.dump(results, f, indent=4)
    print("Final results saved to ipc_sections_fewshot.json")

if __name__ == "__main__":
    main()

Processed 2008.INSC.11.txt
Identified sections: ['Indian Penal Code 498A', 'Indian Penal Code 201']

Processed 2012.INSC.535.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201', 'Indian Penal Code 498A']

Processed 2003.INSC.624.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201']

Processed 1997.INSC.784.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 147', 'Indian Penal Code 506']

Processed 1996.INSC.1253.txt
Identified sections: ['Indian Penal Code 201']

Processed 1996.INSC.1218.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201', 'Indian Penal Code 506']

Processed 2009.INSC.355.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 201', 'Indian Penal Code 147']

Processed 2014.INSC.683.txt
Identified sections: ['Indian Penal Code 420']

Processed 2013.INSC.348.txt
Identified sections: ['Indian Penal Code 302', 'Indian Penal Code 147']

Processed 2002.INSC.461.txt
Identified

COT code

In [ ]:
from groq import Groq
import time
import json
import pandas as pd
import re

# Initialize Groq client
groq_client = Groq(api_key="")

# Load JSON file
with open('/content/test_collection_IPC_147.json', 'r') as f:
    data = json.load(f)

# Process first 5 documents
case_data = []
for filename, case_details in list(data.items())[:5]:
    case_data.append({
        "doc_id": filename,
        "fact": case_details.get("fact", "")
    })

test_df = pd.DataFrame(case_data)

def model_groq(model_name, num_tokens, sp, up):
    while True:
        try:
            completion = groq_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": sp}, {"role": "user", "content": up}],
                temperature=0,
                max_tokens=num_tokens,
                top_p=0.0,
                stream=False,
                stop=None,
            )
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 2 minutes...")
            time.sleep(120)
    return completion.choices[0].message.content

model_name = "llama-3.1-70b-instant"
num_tokens = 6000

prompt = """You are an expert in Indian Penal Code (IPC) statutes. Analyze the legal case fact pattern and:

1. Identify ALL applicable IPC sections from the list below
2. For EACH applicable section:
   - Explain how these phrases satisfy the legal requirements of the section
3. Format your response as a JSON list with:
   - "statute": Full IPC section name
   - "legal_reasoning": Explanation of how phrases match the law

IPC Sections:
"Indian Penal Code 498A": " Whoever, being the husband or the relative of the husband of a woman, subjects such woman to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine.",
"Indian Penal Code 506": " Whoever commits the offence of criminal intimidation shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both",
"Indian Penal Code 147": " Whoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.",
"Indian Penal Code 201": " Whoever, knowing or having reason to believe that an offence has been committed, causes any evidence of the commission of that offence to disappear, with the intention of screening the offender from legal punishment, or with that intention gives any information respecting the offence which he knows or believes to be false;",
"Indian Penal Code 302": " Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 376": " Whoever, except in the cases provided for in sub-section (2), commits rape, shall be punished with rigorous imprisonment of either description for a term which shall not be less than ten years, but which may extend to imprisonment for life, and shall also be liable to fine.",
"Indian Penal Code 420": " Whoever cheats and thereby dishonestly induces the person deceived to deliver any property to any person, or to make, alter or destroy the whole or any part of a valuable security, or anything which is signed or sealed, and which is capable of being converted into a valuable security, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine."


Example Response:
```json
[
    {   "fact": ""
        "statute": "Indian Penal Code 302",
       
        "legal_reasoning": "We have perused the evidence adduced by the prosecution in this case and we notice that though it is true that there was a love affair between Kamla and the appellant, on the date of incident the appellant alongwith 5 other persons did come in tempo and tried to kidnap Kamla at about 10 P.M. and it is because of the intervention of the mother and maternal uncle of the victim alongwith the neighbors, the appellant and another accused by name Kamlesh were apprehended and were produced before the police promptly."
    }
]
```"""

def format_response(response):
    # First try to extract JSON from markdown code block
    json_match = re.search(r'```json\n(.*?)\n```', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Fallback to direct JSON parsing
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        # Final fallback to regex parsing
        pattern = r'''
        \{\s*
            "statute":\s*"Indian Penal Code (302|147|376|498A|506|201|420)",?
            \s*"trigger_phrases":\s*\[(.*?)\],?
            \s*"legal_reasoning":\s*"(.*?)"
        \s*\}'''
        matches = re.findall(pattern, response, re.DOTALL | re.VERBOSE)

        formatted = []
        for match in matches:
            statute_num = match[0]
            phrases = [p.strip().strip('"') for p in match[1].split(',') if p.strip()]
            reasoning = match[2].strip()

            formatted.append({
                "statute": f"Indian Penal Code {statute_num}",
                "trigger_phrases": phrases,
                "legal_reasoning": reasoning
            })

        # Remove duplicates while preserving order
        seen = set()
        unique = []
        for item in formatted:
            identifier = item["statute"]
            if identifier not in seen:
                seen.add(identifier)
                unique.append(item)
        return unique

def main():
    results = {}

    for i in range(min(5, len(test_df))):
        doc_id = test_df.iloc[i]['doc_id']
        fact_text = test_df.iloc[i]['fact']

        response = model_groq(model_name, num_tokens,
                            prompt,
                            f"Case Facts: {fact_text}")

        try:
            sections = format_response(response)
            results[doc_id] = {
                "fact": fact_text,  # Store original fact text
                "analysis": sections,
                "full_response": response  # Keep raw response
            }
        except Exception as e:
            print(f"Error processing {doc_id}: {str(e)}")
            results[doc_id] = {
                "fact": fact_text,
                "error": str(e),
                "full_response": response
            }

        print(f"\nProcessed {doc_id}")
        print("Identified Sections:")
        for section in results[doc_id].get('analysis', []):
            print(f"\nSection: {section['statute']}")
            print("Trigger Phrases:", ", ".join(f'"{p}"' for p in section['trigger_phrases']))
            print("Reasoning:", section['legal_reasoning'])
        print("-" * 50)

    # Save results
    with open("ipc_sections_detailed_analysis.json", "w") as f:
        json.dump(results, f, indent=4)
    print("\nFinal results saved to ipc_sections_detailed_analysis.json")

if __name__ == "__main__":
    main()


Processed 2004.INSC.496.txt
Identified Sections:

Section: Indian Penal Code 147
Trigger Phrases: "alongwith 5 other persons did come in tempo and tried to kidnap Kamla"
Reasoning: The phrase 'alongwith 5 other persons did come in tempo and tried to kidnap Kamla' indicates a group of people, including the appellant, engaging in a collective act of rioting, which satisfies IPC 147's definition of rioting.

Section: Indian Penal Code 506
Trigger Phrases: "tried to kidnap Kamla"
Reasoning: The phrase 'tried to kidnap Kamla' indicates an act of criminal intimidation, as the appellant and others attempted to forcibly take Kamla away from her lawful guardianship, satisfying IPC 506's definition of criminal intimidation.

Section: Indian Penal Code 201
Trigger Phrases: "a false complaint was lodged against the appellant and others"
Reasoning: The phrase 'a false complaint was lodged against the appellant and others' indicates an attempt to cause evidence of the commission of an offence to di